In [0]:
base = "/Volumes/workspace/default/attendance"

attendance = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/attendance_large.csv")
tasks = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/tasks_large.csv")

print("Attendance rows:", attendance.count())
print("Tasks rows:", tasks.count())
display(attendance.limit(5))

Attendance rows: 108500
Tasks rows: 108500


employee_id,department,work_date,clock_in,clock_out,break_minutes
1,Support,2026-01-01,2026-01-01T09:07:00.000Z,2026-01-01T16:12:40.570Z,60
1,Support,2026-01-02,2026-01-02T08:53:00.000Z,2026-01-02T17:13:20.459Z,45
1,Support,2026-01-05,2026-01-05T09:28:00.000Z,2026-01-05T17:10:16.334Z,45
1,Support,2026-01-06,2026-01-06T09:02:00.000Z,2026-01-06T16:08:41.254Z,60
1,Support,2026-01-07,2026-01-07T09:27:00.000Z,2026-01-07T17:48:06.293Z,45


In [0]:
from pyspark.sql import functions as F

attendance_t = (attendance
    .withColumn("clock_in", F.to_timestamp("clock_in"))
    .withColumn("clock_out", F.to_timestamp("clock_out"))
    .withColumn("break_minutes", F.col("break_minutes").cast("double"))
    .withColumn("break_minutes",
                F.when(F.col("break_minutes") < 0, 0)
                 .otherwise(F.col("break_minutes")))
    .fillna({"break_minutes": 0})
    .dropDuplicates(["employee_id", "work_date"])
    .dropna(subset=["employee_id", "work_date"])
    .withColumn("is_absent",
                F.col("clock_in").isNull() & F.col("clock_out").isNull())
    .withColumn("work_hours",
                F.when(F.col("is_absent"), 0)
                 .otherwise(
                    (F.unix_timestamp("clock_out") - F.unix_timestamp("clock_in")) / 3600
                    - F.col("break_minutes") / 60))
    .withColumn("is_late",
                F.when(F.col("clock_in").isNotNull(),
                    (F.hour("clock_in") * 60 + F.minute("clock_in")) > (9 * 60 + 15))
                 .otherwise(False))
)

print("Rows:", attendance_t.count())
display(attendance_t.limit(5))

Rows: 108500


employee_id,department,work_date,clock_in,clock_out,break_minutes,is_absent,work_hours,is_late
1,Support,2026-01-06,2026-01-06T09:02:00.000Z,2026-01-06T16:08:41.254Z,60.0,false,6.1113888888888885,false
1,Support,2026-01-16,2026-01-16T09:16:00.000Z,2026-01-16T18:06:49.391Z,30.0,false,8.346944444444444,true
1,Support,2026-02-06,2026-02-06T08:59:00.000Z,2026-02-06T17:03:20.359Z,30.0,false,7.572222222222223,false
1,Support,2026-02-13,2026-02-13T08:53:00.000Z,2026-02-13T17:22:09.757Z,45.0,false,7.735833333333334,false
1,Support,2026-03-10,null,null,45.0,true,0.0,false


In [0]:
tasks_t = (tasks
    .withColumn("work_date", F.to_date("work_date"))
    .withColumn("tasks_assigned", F.col("tasks_assigned").cast("int"))
    .withColumn("tasks_completed", F.col("tasks_completed").cast("double"))
    .fillna({"tasks_completed": 0})
)

attendance_enriched = attendance_t.join(tasks_t, on=["employee_id", "work_date"], how="left")

attendance_enriched = attendance_enriched.withColumn(
    "productivity_score",
    F.when(F.col("work_hours") > 0,
           F.round(F.col("tasks_completed") / F.col("work_hours"), 2))
     .otherwise(0)
)

print("Rows after join:", attendance_enriched.count())
print("Rows missing tasks:", attendance_enriched.filter(F.col("tasks_completed").isNull()).count())

display(attendance_enriched.limit(5))

Rows after join: 108500
Rows missing tasks: 0


employee_id,work_date,department,clock_in,clock_out,break_minutes,is_absent,work_hours,is_late,tasks_assigned,tasks_completed,productivity_score
1,2026-01-06,Support,2026-01-06T09:02:00.000Z,2026-01-06T16:08:41.254Z,60.0,false,6.1113888888888885,false,3,2.0,0.33
1,2026-01-16,Support,2026-01-16T09:16:00.000Z,2026-01-16T18:06:49.391Z,30.0,false,8.346944444444444,true,5,2.0,0.24
1,2026-02-06,Support,2026-02-06T08:59:00.000Z,2026-02-06T17:03:20.359Z,30.0,false,7.572222222222223,false,6,2.0,0.26
1,2026-02-13,Support,2026-02-13T08:53:00.000Z,2026-02-13T17:22:09.757Z,45.0,false,7.735833333333334,false,4,2.0,0.26
1,2026-03-10,Support,null,null,45.0,true,0.0,false,3,3.0,0.0


In [0]:
dept_metrics = (attendance_enriched
    .groupBy("department")
    .agg(
        F.round(F.mean("work_hours"), 2)        .alias("avg_work_hours"),
        F.round(F.mean("productivity_score"), 2).alias("avg_productivity"),
        F.round(F.sum("work_hours"), 2)         .alias("total_work_hours"),
        F.sum(F.col("is_absent").cast("int"))   .alias("total_absences"),
        F.sum(F.col("is_late").cast("int"))     .alias("total_late_logins"),
        F.count("employee_id")                  .alias("total_records")
    )
    .withColumn("absence_rate_%",
        F.round(F.col("total_absences") / F.col("total_records") * 100, 2))
    .withColumn("late_rate_%",
        F.round(F.col("total_late_logins") / F.col("total_records") * 100, 2))
    .orderBy("department")
)

print("Departments:", dept_metrics.count())
display(dept_metrics)

Departments: 5


department,avg_work_hours,avg_productivity,total_work_hours,total_absences,total_late_logins,total_records,absence_rate_%,late_rate_%
Engineering,6.89,0.3,152485.54,1104,7760,22134,4.99,35.06
Finance,6.88,0.3,153788.46,1158,7663,22351,5.18,34.28
HR,6.88,0.3,126823.47,949,6361,18445,5.15,34.49
Sales,6.89,0.3,153907.69,1114,7828,22351,4.98,35.02
Support,6.9,0.3,160317.53,1118,8166,23219,4.82,35.17


In [0]:
(attendance_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.attendance_enriched"))

(dept_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.department_metrics"))

print("Rows saved (attendance_enriched):", spark.table("workspace.default.attendance_enriched").count())
print("Rows saved (department_metrics):", spark.table("workspace.default.department_metrics").count())

display(spark.table("workspace.default.department_metrics"))

Rows saved (attendance_enriched): 108500
Rows saved (department_metrics): 5


department,avg_work_hours,avg_productivity,total_work_hours,total_absences,total_late_logins,total_records,absence_rate_%,late_rate_%
Engineering,6.89,0.3,152485.54,1104,7760,22134,4.99,35.06
Finance,6.88,0.3,153788.46,1158,7663,22351,5.18,34.28
HR,6.88,0.3,126823.47,949,6361,18445,5.15,34.49
Sales,6.89,0.3,153907.69,1114,7828,22351,4.98,35.02
Support,6.9,0.3,160317.53,1118,8166,23219,4.82,35.17


In [0]:
%sql
SELECT 
    department,
    avg_work_hours,
    avg_productivity,
    total_absences,
    total_late_logins,
    `absence_rate_%`,
    `late_rate_%`
FROM workspace.default.department_metrics
ORDER BY `absence_rate_%` DESC

department,avg_work_hours,avg_productivity,total_absences,total_late_logins,absence_rate_%,late_rate_%
Finance,6.88,0.3,1158,7663,5.18,34.28
HR,6.88,0.3,949,6361,5.15,34.49
Engineering,6.89,0.3,1104,7760,4.99,35.06
Sales,6.89,0.3,1114,7828,4.98,35.02
Support,6.9,0.3,1118,8166,4.82,35.17


In [0]:
import os

os.makedirs(f"{base}/output", exist_ok=True)

dept_metrics.toPandas().to_csv(f"{base}/output/department_kpis.csv", index=False)

print("Saved department_kpis.csv!")
display(dbutils.fs.ls(f"{base}/output"))

Saved department_kpis.csv!


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/attendance/output/department_kpis.csv,department_kpis.csv,399,1794916339000
